In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *

###Regras de qualidade dos clientes


####1 - Quantos registros agente tem

In [0]:
def contar_registros(df: DataFrame):
    print("Quantidade de registros: %d\n" % df.count())
    return df

####2 - A estrutura ou schema da tabela cliente


In [0]:
def estrutura(df: DataFrame):
    print(f"Estrutura da tabela possui {len(df.columns)}\n")
    df.printSchema()
    return df

####3 - Plenitude dos dados

In [0]:
def plenitude(df: DataFrame):
    print("Verificando plenitude dos dados em cada coluna\n")
    df_nulos=df.select(
        *[
            count(when(col(c).isNull(), c)).alias(c)
            for c in df.columns
        ]
    )
    df_nulos.show()
    return df

####4 - Obrigatoriedade dos dados

In [0]:
def obrigatoriedade(df: DataFrame):
    print("Verificando os campos obrigatorios que nao podem ter nullos\n")
    obr = [
        "id",
        "primeiro_nome",
        "ultimo_nome",
        "data_nascimento",
        "data_criacao",
        "municipio",
        "provincia",
        "rendimento_mensal"
    ]
    
    for c in obr:
        print(f"O campo {c} tem {df.filter(col(c).isNull()).count()} de nullos")
    return df


####5 - unicidade

In [0]:
def unicidade(df: DataFrame):
    print("Verificando unicidade dos dados em colulunas que devem ser unicas\n")
    uni = [
        "id",
        "bi",
        "nif",
        "email"    
    ]
    """# ID
    duplicados_id = (
        df.filter(col("id").isNotNull())
        .groupBy("id")
        .count()
        .filter(col("count") > 1)
    )

    print("Duplicados em id:")
    duplicados_id.show()"""
    for c in uni:
        df_duplicados = df.filter(col(c).isNotNull()) \
            .groupBy(c) \
                .count() \
                .filter(col("count") > 1)
        df_duplicados.show(truncate=False)
        print(f"O campo {c} tem {df_duplicados.count()} duplicados")
        print("\n")

    return df

####6 - validacao dos dados

In [0]:
def validacao(df: DataFrame):
    df_v = df.groupBy("estado_civil") \
            .count() \
            .orderBy(col("count").desc())
    #df_v.show(truncate=False)
    """#df_r = df.select("rendimento_mensal").summary().show()
    #df_r = df.orderBy(col("rendimento_mensal").desc()).limit(10).select("id", "rendimento_mensal").show()
    #           ver data de nascimento maior que data atual
    #df_nas = df.select("id", "data_nascimento").filter(col("data_nascimento") > current_date()).show()
    # ver data de criacao menor que data atual
    #df_cri = df.select("id", "data_criacao").filter(col("data_criacao") > current_date()).show()
    df_nas = df.select(
            min("data_nascimento").alias("data_minima"),
            max("data_nascimento").alias("data_maxima")
        ).show()
    # investigas emails
    df_email = df.filter(
        (col("email").isNotNull()) &
        (col("email") != "") &
        (
            ~col("email").contains("@")
        )
        # ou (~col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$"))
    )
    df_email.select("id", "email").show()
    # investigar telefone
    df_telefone = df_cliente.filter(
    (col("telefone").isNotNull()) &
    (trim(col("telefone")) != "") &
    (~col("telefone").rlike(r"^[0-9+\s-]+$"))
    )

    df_telefone.select(
    "id",
    "telefone"
    ).show(truncate=False)"""

    d = df.filter(col("data_nascimento") > col("data_criacao")).select("id", "data_nascimento", "data_criacao").collect()
    if len(d) > 0:
        print("Data de nascimento maior que data de criacao")
    return df

In [0]:
def qualidade_dados(df: DataFrame):
    """df = contar_registros(df)
    df = estrutura(df)
    df = plenitude(df)
    df = obrigatoriedade(df)"""
    df = unicidade(df)
    """df = validacao(df)"""

    return df

In [0]:
df_cliente = spark.read.table("dbw_banco_orion.bronze.clientes")
df_cliente = qualidade_dados(df_cliente)